In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. 심화 전처리 및 K-Means 군집화 (완벽 방어)
# ==========================================
def preprocess_and_cluster(train_path, layout_path):
    print("1. 데이터 로드 및 심화 전처리 시작...")
    train = pd.read_csv(train_path)
    layout = pd.read_csv(layout_path)
    
    train = pd.merge(train, layout, on='layout_id', how='left')
    train.drop('ID', axis=1, inplace=True)

    target_col = 'avg_delay_minutes_next_30m'
    y_train = train[target_col]
    X_train = train.drop(target_col, axis=1)
    
    # 결측치 처리
    for col in X_train.select_dtypes(include=['float64', 'int64']).columns:
        X_train[col].fillna(X_train[col].median(), inplace=True)

    # 파생 변수 (주기성 원본 삭제)
    X_train['shift_hour_sin'] = np.sin(2 * np.pi * X_train['shift_hour'] / 24)
    X_train['shift_hour_cos'] = np.cos(2 * np.pi * X_train['shift_hour'] / 24)
    X_train['day_of_week_sin'] = np.sin(2 * np.pi * X_train['day_of_week'] / 7)
    X_train['day_of_week_cos'] = np.cos(2 * np.pi * X_train['day_of_week'] / 7)
    X_train.drop(['shift_hour', 'day_of_week'], axis=1, inplace=True) 
    
    if 'robot_utilization' in X_train.columns and 'congestion_score' in X_train.columns:
        X_train['robot_util_x_congestion'] = X_train['robot_utilization'] * X_train['congestion_score']
        
    X_train = pd.get_dummies(X_train, columns=['layout_type'], drop_first=False)

    print("2. 창고 상태(쾌적/혼잡/마비) 3분할 군집화 진행 중...")
    possible_features = ['robot_utilization', 'congestion_score', 'charge_queue_length', 'task_reassign_15m']
    cluster_features = [f for f in possible_features if f in X_train.columns]
    
    X_cluster = X_train[cluster_features].copy()
    X_cluster.replace([np.inf, -np.inf], np.nan, inplace=True)
    X_cluster.fillna(0, inplace=True)
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_cluster)
    
    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    X_train['cluster_id'] = kmeans.fit_predict(X_scaled)
    
    return X_train, y_train

# ==========================================
# 2. 군집별 맞춤형 앙상블 학습
# ==========================================
def validate_cluster_ensemble(X_train, y_train):
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    oof_predictions = np.zeros(len(X_train))
    
    # ----------------------------------------------------
    # [핵심] 군집별 맞춤형 파라미터 딕셔너리
    # ----------------------------------------------------
    # Cluster 0 (쾌적): 얕은 트리, 빠른 학습
    params_c0 = {
        'lgb': {'objective': 'mae', 'n_estimators': 1500, 'learning_rate': 0.05, 'max_depth': 6, 'num_leaves': 31, 'n_jobs': -1, 'verbose': -1},
        'xgb': {'objective': 'reg:absoluteerror', 'n_estimators': 1500, 'learning_rate': 0.05, 'max_depth': 6, 'n_jobs': -1, 'early_stopping_rounds': 50},
        'cat': {'loss_function': 'MAE', 'iterations': 1500, 'learning_rate': 0.05, 'depth': 6, 'verbose': False, 'early_stopping_rounds': 50},
        'weights': (0.5, 0.3, 0.2) # LGBM 중심
    }
    
    # Cluster 1 (혼잡): 스탠다드 밸런스
    params_c1 = {
        'lgb': {'objective': 'mae', 'n_estimators': 1500, 'learning_rate': 0.04, 'max_depth': 8, 'num_leaves': 63, 'n_jobs': -1, 'verbose': -1},
        'xgb': {'objective': 'reg:absoluteerror', 'n_estimators': 1500, 'learning_rate': 0.04, 'max_depth': 8, 'n_jobs': -1, 'early_stopping_rounds': 50},
        'cat': {'loss_function': 'MAE', 'iterations': 1500, 'learning_rate': 0.04, 'depth': 8, 'verbose': False, 'early_stopping_rounds': 50},
        'weights': (0.4, 0.3, 0.3) # 균형
    }
    
    # Cluster 2 (마비): 깊은 트리, 노이즈에 강한 CatBoost 비중 상향
    params_c2 = {
        'lgb': {'objective': 'mae', 'n_estimators': 1500, 'learning_rate': 0.03, 'max_depth': 10, 'num_leaves': 127, 'n_jobs': -1, 'verbose': -1},
        'xgb': {'objective': 'reg:absoluteerror', 'n_estimators': 1500, 'learning_rate': 0.03, 'max_depth': 10, 'n_jobs': -1, 'early_stopping_rounds': 50},
        'cat': {'loss_function': 'MAE', 'iterations': 2000, 'learning_rate': 0.03, 'depth': 10, 'verbose': False, 'early_stopping_rounds': 50},
        'weights': (0.3, 0.3, 0.4) # CatBoost 중심
    }
    
    cluster_configs = {0: params_c0, 1: params_c1, 2: params_c2}
    
    print("\n3. 군집별 3-Way 맞춤형 앙상블 K-Fold 검증 시작...")
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
        x_tr, y_tr = X_train.iloc[train_idx].copy(), y_train.iloc[train_idx]
        x_val, y_val = X_train.iloc[val_idx].copy(), y_train.iloc[val_idx]
        
        overall_mean = y_tr.mean()
        
        # Target Encoding
        for col in ['scenario_id', 'layout_id']:
            if col in x_tr.columns:
                group_stats = y_tr.groupby(x_tr[col]).agg(mean=('mean'), count=('count'))
                smoothed_means = (group_stats['mean'] * group_stats['count'] + overall_mean * 10) / (group_stats['count'] + 10)
                x_tr[f'{col}_target_enc'] = x_tr[col].map(smoothed_means)
                x_val[f'{col}_target_enc'] = x_val[col].map(smoothed_means).fillna(overall_mean)
                x_tr.drop(col, axis=1, inplace=True)
                x_val.drop(col, axis=1, inplace=True)

        fold_preds = np.zeros(len(x_val))
        
        # 3개 군집별로 각기 다른 모델과 가중치로 학습!
        for cid in range(3):
            tr_mask = x_tr['cluster_id'] == cid
            val_mask = x_val['cluster_id'] == cid
            
            if not val_mask.any() or not tr_mask.any(): continue
            
            x_tr_c = x_tr[tr_mask].drop('cluster_id', axis=1)
            y_tr_c = y_tr[tr_mask]
            x_val_c = x_val[val_mask].drop('cluster_id', axis=1)
            y_val_c = y_val[val_mask]
            
            cfg = cluster_configs[cid]
            w_lgb, w_xgb, w_cat = cfg['weights']
            
            # 1. LGBM
            model_lgb = lgb.LGBMRegressor(**cfg['lgb'], random_state=42)
            model_lgb.fit(x_tr_c, y_tr_c, eval_set=[(x_val_c, y_val_c)], callbacks=[lgb.early_stopping(50, verbose=False)])
            pred_lgb = model_lgb.predict(x_val_c)
            
            # 2. XGBoost
            model_xgb = xgb.XGBRegressor(**cfg['xgb'], random_state=42)
            model_xgb.fit(x_tr_c, y_tr_c, eval_set=[(x_val_c, y_val_c)], verbose=False)
            pred_xgb = model_xgb.predict(x_val_c)
            
            # 3. CatBoost
            model_cat = cb.CatBoostRegressor(**cfg['cat'], random_seed=42)
            model_cat.fit(x_tr_c, y_tr_c, eval_set=(x_val_c, y_val_c), use_best_model=True)
            pred_cat = model_cat.predict(x_val_c)
            
            # 군집 전용 가중치 블렌딩
            fold_preds[val_mask] = (pred_lgb * w_lgb) + (pred_xgb * w_xgb) + (pred_cat * w_cat)
            
        oof_predictions[val_idx] = fold_preds
        print(f"  - Fold {fold+1} 앙상블 완료 | MAE: {mean_absolute_error(y_val, fold_preds):.4f}")
        
    total_mae = mean_absolute_error(y_train, oof_predictions)
    print("\n" + "="*50)
    print(f"🏆 [군집별 3-Way 앙상블] 최종 OOF MAE: {total_mae:.4f}")
    print("="*50 + "\n")

# ==========================================
# 3. 메인 실행부
# ==========================================
if __name__ == "__main__":
    TRAIN_PATH = 'train.csv' # 환경에 맞춰 수정
    LAYOUT_PATH = 'layout_info.csv'
    
    X_train, y_train = preprocess_and_cluster(TRAIN_PATH, LAYOUT_PATH)
    validate_cluster_ensemble(X_train, y_train)

1. 데이터 로드 및 심화 전처리 시작...
2. 창고 상태(쾌적/혼잡/마비) 3분할 군집화 진행 중...

3. 군집별 3-Way 맞춤형 앙상블 K-Fold 검증 시작...
  - Fold 1 앙상블 완료 | MAE: 4.7365


KeyboardInterrupt: 